# Medium-Effort Optimization Playbook

The **six MEDIUM-effort levers** — architectural moves on well-trodden paths, a step up from the one-line tweaks of the LOW tier.

| # | Lever | What it does |
|---|---|---|
| 1 | LLM routing | Classify each request, send it to the cheapest model that can handle it |
| 2 | Bedrock Guardrails | Block / redact at the API layer — refused requests never cost generation |
| 3 | RAG / indexing | Retrieve only the top-K chunks the query needs, not the whole corpus |
| 4 | Prompt compression | Strip low-signal tokens from the dynamic context before the model sees it |
| 5 | Conversation & memory | Bound per-turn context (sliding window + summary), persist facts across sessions |
| 6 | Batch inference | Run offline work asynchronously at 50% of the on-demand price |

Each lever is independent — lift any one into your own application without the others. Run the setup cell, then work through them in order.

## Setup

In [ ]:
import boto3, time, json
REGION = "us-east-1"
RUNTIME = boto3.client("bedrock-runtime", region_name=REGION)

HAIKU = "global.anthropic.claude-haiku-4-5-20251001-v1:0"
SONNET = "global.anthropic.claude-sonnet-4-6"

# Mock customer-support tools (order status, return policy, product info) so the
# agents below answer real questions instead of "I'm an AI, I can't look that up".
# Hardcoded data — the point is to exercise tool-calling realistically (see utils/support_tools.py).
from strands import Agent
from strands.models import BedrockModel
from utils.support_tools import SUPPORT_TOOLS

def support_agent(model_id, system_prompt="You are a customer-support agent. Use your tools to look "
                  "up real order, policy, and product info before answering. Be concise."):
    """A Strands agent wired to the mock support tools, on the given model."""
    return Agent(model=BedrockModel(model_id=model_id, region_name=REGION),
                 tools=SUPPORT_TOOLS, system_prompt=system_prompt, callback_handler=None)

## 1. LLM Routing — Lever 07

Query complexity has a long tail. In a typical workload only ~20% of requests genuinely need a workhorse like Sonnet 4.6 or Opus 4.8; the other ~80% are factual lookups or single-step classifications Haiku 4.5 answers just as well at ~**1/3 the cost**. Routing classifies each request, then sends it to the *cheapest* model that can handle it.

There's a spectrum of how you classify, trading setup effort against routing quality:

| Pattern | How it works | Latency | Quality |
|---|---|---|---|
| Static rules | Token count / language / code-fence → bucket | Sub-ms | Brittle but transparent |
| **LLM classifier** | Cheap LLM emits a complexity/intent label | ~100-300ms | Good if the prompt is good |
| Cascade (FrugalGPT) | Cheap model first; escalate on low confidence | Variable | Best case = small-model only |
| Preference classifier (RouteLLM) | Trained on human preference data | Sub-ms | ~95% strong-model quality at ~26% cost |
| Generative router (Arch-Router) | Small LM emits a route token | ~50-150ms | SOTA on multi-turn agents |

We build the **LLM-classifier** pattern below — the practical starting point. Two rules keep it a win:

- **Classifier must be tiny and deterministic** — `temperature=0`, `maxTokens≈5`, and **cache its static prompt** (Lever 04).
- **The cheap path must take the majority of traffic** — and you must confirm routed-down quality on your own eval set, not the vendor's benchmark.

The failure mode to watch is the floor: if the classifier round-trip makes the "simple" path slower or pricier than just calling the big model, routing has cost you.

In [ ]:
def classify_complexity(query: str) -> str:
    # Few-shot examples pin the output to one word; the "###" stop sequence ends
    # generation right after it, so the classifier costs ~4 output tokens, not a sentence.
    # The classifier stays tiny and TOOL-FREE — it only labels; the answering agent does the work.
    resp = RUNTIME.converse(
        modelId=HAIKU,
        messages=[{"role": "user", "content": [{"text": (
            "Classify the customer query as exactly one word: 'simple' (factual lookup, "
            "single step) or 'complex' (multi-step reasoning, ambiguous, or policy "
            "interpretation). Output only the word.\n\n"
            "Query: What are your store hours?\nClassification: simple ###\n"
            "Query: I returned 2 of 3 bundled items; what refund is owed on the third?\n"
            "Classification: complex ###\n"
            f"Query: {query}\nClassification:"
        )}]}],
        inferenceConfig={"maxTokens": 5, "temperature": 0, "stopSequences": ["###"]},
    )
    return resp["output"]["message"]["content"][0]["text"].strip().lower()

def route_and_call(query: str):
    label = classify_complexity(query)
    target = HAIKU if "simple" in label else SONNET
    # The answering agent is tool-backed (order status, return policy, product info), so it
    # gives grounded answers — the only thing routing changes is WHICH model runs it.
    agent = support_agent(target)
    answer = str(agent(query))
    return {"label": label, "routed_to": target.split(".")[-1], "text": answer}

test_queries = [
    "Where is my order #12345?",                                            # simple — order lookup
    "What's your return policy on opened headphones?",                       # simple — policy lookup
    "My laptop arrived damaged but it's been 35 days; what can you do?",     # complex (policy edge case)
    "I bought 3 phones, returned 2 last week, want to return the third now "
    "but the original receipt says they were a bundle deal — refund amount?",  # complex
]

for q in test_queries:
    r = route_and_call(q)
    print(f"\nQ ({r['label']}, routed to {r['routed_to']}):")
    print(f"  {q}")
    print("A:")
    print(r["text"])

**What you'll see**: ~80% of typical queries route to Haiku → ~3× cheaper for those, while the classifier adds only ~4 output tokens. The simple lookups ("where's my order", "return policy") land on Haiku and answer straight from the order/policy tools; the policy edge-cases escalate to Sonnet, which reasons over the same tool results. Routing changes only *which* model runs — both have the tools.

---

## 2. Bedrock Guardrails — Lever 08

Guardrails are sold as **safety** — block toxic prompts, refuse off-scope topics, redact PII. They're equally a **cost-and-latency** lever: a request blocked at the input layer never reaches the model, so you don't pay generation tokens on injection probes, jailbreaks, or off-topic traffic. On any public endpoint, abuse traffic is constant, and an off-topic "write me a poem" costs the same tokens as a real question.

A guardrail bundles **independent** policies, evaluated separately, applied differently to **input** vs **output**:

| Policy | Catches | Action |
|---|---|---|
| Content filters | Hate, insults, sexual, violence, misconduct (text + images) | Strength LOW/MED/HIGH per category |
| Prompt-attack detection | Jailbreak / prompt-injection (input-side) | Block |
| Denied topics | Off-scope subjects defined in natural language | Block |
| Word filters | Custom block lists + managed profanity | Block |
| Sensitive info (PII) | Email, phone, SSN, cards, names + your regex | **BLOCK** or **ANONYMIZE** |
| Contextual grounding | RAG hallucinations — scores grounding + relevance | Block below threshold |

We demo two ways to call it: **inline** with `converse` (one guardrailed call), then the **standalone `ApplyGuardrail` API** — the cost trick that screens text *without invoking a model*, stopping bad traffic before it costs you retrieval **and** generation.

<div class="alert alert-block alert-info">

**Pricing is per "text unit" ≈ 1,000 characters**, billed per policy evaluated — so blocking 5% off-topic traffic at the input layer saves ~5% of your *full* request cost (retrieval + generation), which more than funds the eval fee. Batch screening in 1,000-char multiples to avoid waste.

</div>

In [ ]:
BEDROCK = boto3.client("bedrock", region_name=REGION)

# Create a minimal guardrail (ephemeral — cleaned up at end of this section)
guardrail = BEDROCK.create_guardrail(
    name=f"playbook-workshop-{int(time.time())}",
    description="Workshop demo — blocks competitor questions and redacts PII",
    topicPolicyConfig={
        "topicsConfig": [
            {
                "name": "Competitor questions",
                "definition": "Questions comparing us to or recommending named competitors",
                "examples": ["Why is the other store cheaper?", "Should I shop somewhere else instead?"],
                "type": "DENY",
            }
        ]
    },
    sensitiveInformationPolicyConfig={
        "piiEntitiesConfig": [
            {"type": "EMAIL", "action": "ANONYMIZE"},
            {"type": "PHONE", "action": "ANONYMIZE"},
            {"type": "CREDIT_DEBIT_CARD_NUMBER", "action": "BLOCK"},
        ]
    },
    blockedInputMessaging="Sorry, I can't answer questions about competitors.",
    blockedOutputsMessaging="Output blocked by safety policy.",
)
GUARDRAIL_ID = guardrail["guardrailId"]
GUARDRAIL_VERSION = "DRAFT"
print(f"Created guardrail: {GUARDRAIL_ID}")

In [ ]:
def converse_with_guardrail(query):
    return RUNTIME.converse(
        modelId=SONNET,
        messages=[{"role": "user", "content": [{"text": query}]}],
        inferenceConfig={"maxTokens": 300, "temperature": 0},
        guardrailConfig={
            "guardrailIdentifier": GUARDRAIL_ID,
            "guardrailVersion": GUARDRAIL_VERSION,
            "trace": "enabled",
        },
    )

for q in [
    "How does your warranty compare to the store down the street?",  # should be blocked (competitor)
    "My email is alice@example.com — can you confirm receipt?",      # PII anonymized
    "What's your warranty on a refurbished laptop?",                  # should pass
]:
    r = converse_with_guardrail(q)
    stop = r.get("stopReason", "end_turn")
    text = r["output"]["message"]["content"][0].get("text", "<no text>")
    print(f"\n[{stop}] {q}")
    print(f"  -> {text}")

**`ApplyGuardrail` standalone — screen input without calling the model.** The inline `guardrailConfig` above attaches the guardrail to a Converse call: input is screened *before* the model runs, so a blocked request never pays for inference either. The **standalone `ApplyGuardrail` API** does the same screening as its own call — the difference isn't cost, it's **where you put it in the pipeline**:

- Run one guardrail in front of *any* model — including non-Bedrock or self-hosted — not just Bedrock Converse calls.
- Screen at an earlier stage (before retrieval, routing, or tool calls), so bad traffic is rejected before *any* downstream work, not just before the LLM.
- Reuse a single guardrail definition across several models/endpoints in your stack.

Either way a blocked request avoids the model spend; standalone just lets you place that check wherever it fits your architecture.

In [ ]:
# Standalone ApplyGuardrail: screen INPUT with NO model invocation.
for label, text in [
    ("competitor", "Should I just shop at the store down the street instead?"),  # denied topic
    ("legitimate", "What's your warranty on a refurbished laptop?"),             # passes
]:
    r = RUNTIME.apply_guardrail(
        guardrailIdentifier=GUARDRAIL_ID,
        guardrailVersion=GUARDRAIL_VERSION,
        source="INPUT",
        content=[{"text": {"text": text}}],
    )
    print(f"[{label:11}] action={r['action']:22} (no model called)")
    if r["action"] == "GUARDRAIL_INTERVENED":
        print("              -> refused for the price of one guardrail eval, not a full inference")

In [ ]:
BEDROCK.delete_guardrail(guardrailIdentifier=GUARDRAIL_ID)
print(f"Deleted guardrail {GUARDRAIL_ID}")

**What you'll see**: the competitor question returns `stopReason=guardrail_intervened` (inline) and `GUARDRAIL_INTERVENED` (standalone) — no real answer generated, so it bills a fraction of a full inference. The PII email is anonymized in place; legitimate queries pass. At a 5% probe rate, those early refusals are real savings.

---

## 3. RAG / Indexing — Lever 09

The instinct on a large knowledge source is to stuff it all into context. That's expensive *and* counterproductive: past a certain size accuracy drops because the model underweights information buried mid-prompt (the "lost in the middle" effect). RAG inverts it — embed the corpus once, then per query retrieve only the top-K relevant chunks. The cost win is the retrieve step: 3-5K tokens of relevant context instead of a 50K-token dump.

Four levers carry most of the win, and we run them against a **dedicated playbook Knowledge Base** — the workshop stack builds it, uploads docs with per-document metadata, and **ingests it for you** (separate from the Developer-Journey KB), so it's queryable the moment you arrive:

| Lever | What it does | We demo |
|---|---|---|
| **Chunking strategy** | How docs are split — dominates retrieval quality | Read back the KB's config (set on the data source) |
| **top-K tuning** | How many chunks you pass | `numberOfResults` 10 → 5 → 3 |
| **Metadata filtering** | Filter at query time (product line, doc type) | `filter` on `retrieve` |
| **Reranking** | Cross-encoder reorders candidates, best chunk first | `rerankingConfiguration` on `retrieve` |

<div class="alert alert-block alert-warning">
<b>Cache interaction:</b> retrieved chunks are <b>dynamic</b> — keep them <i>after</i> your <code>cachePoint</code>. Putting them before the cached prefix busts the cache on every query (Lever 04).
</div>

In [ ]:
# Read the playbook KB id + data-source id from SSM. The workshop stack already
# created the KB, uploaded docs, and RAN INGESTION at deploy time — so it's queryable now.
sts = boto3.client("sts", region_name=REGION)
account_id = sts.get_caller_identity()["Account"]
ssm = boto3.client("ssm", region_name=REGION)

def _param(name):
    return ssm.get_parameter(Name=name)["Parameter"]["Value"]

KB_ID = DS_ID = ""
try:
    KB_ID = _param(f"/{account_id}-{REGION}/playbook-kb/knowledge-base-id")
    DS_ID = _param(f"/{account_id}-{REGION}/playbook-kb/data-source-id")
    print("Playbook KB:", KB_ID, "| data source:", DS_ID)
except Exception as e:
    print("Could not read the playbook KB params from SSM:", e)
    print("Deploy the workshop stack — it creates /{account}-{region}/playbook-kb/*.")

AGENT_RT = boto3.client("bedrock-agent-runtime", region_name=REGION)
AGENT = boto3.client("bedrock-agent", region_name=REGION)
QUERY = "What's the warranty on a refurbished laptop?"

# Inspect the chunking strategy the KB was built with — it lives on the data source.
if KB_ID and DS_ID:
    ds = AGENT.get_data_source(knowledgeBaseId=KB_ID, dataSourceId=DS_ID)
    chunk = ds["dataSource"]["vectorIngestionConfiguration"]["chunkingConfiguration"]
    print("\nChunking strategy:", chunk["chunkingStrategy"])
    print(json.dumps(chunk, indent=2, default=str))
else:
    print("Skipped — no playbook KB params available.")

**What you'll see**: the data source reports `HIERARCHICAL` chunking — parent chunks (~1500 tokens) for context, child chunks (~300 tokens) that are actually searched. This is set on the data source and **applied by the ingestion job**; the two are coupled, so to change chunking you'd update the config and re-ingest.

<div class="alert alert-block alert-info">

**Why hierarchical is the default.** You search the small child chunks (precise vector matches) but hand the model the larger parent (surrounding context) — best of both. Alternatives: <b>semantic</b> (split on embedding-similarity breakpoints — better for fuzzy document boundaries, more compute) and <b>fixed-size</b> (uniform windows — cheap baseline). Start hierarchical; A/B against semantic on your eval set.

</div>

In [ ]:
# --- Lever: top-K tuning ---
# The simplest cost lever and the most over-set. Tune K *down* and watch the context shrink.
if KB_ID:
    for top_k in [10, 5, 3]:
        resp = AGENT_RT.retrieve(
            knowledgeBaseId=KB_ID,
            retrievalQuery={"text": QUERY},
            retrievalConfiguration={"vectorSearchConfiguration": {"numberOfResults": top_k}},
        )
        chunks = resp["retrievalResults"]
        chars = sum(len(c["content"]["text"]) for c in chunks)
        print(f"top_k={top_k:2}  chunks={len(chunks):2}  total_chars={chars:5}")
else:
    print("Skipped — no playbook KB available.")

In [ ]:
# --- Lever: metadata filtering ---
# Each doc was uploaded with a .metadata.json companion ({product_line, doc_type}).
# Filtering at query time shrinks the candidate set BEFORE the vector search —
# better precision, multi-tenant isolation, and less to rank.
if KB_ID:
    # Only policy docs, only the 'laptops' or 'all' product lines.
    flt = {"andAll": [
        {"equals": {"key": "doc_type", "value": "policy"}},
        {"in": {"key": "product_line", "value": ["laptops", "all"]}},
    ]}
    for label, cfg in [("no filter", {}), ("filtered", {"filter": flt})]:
        resp = AGENT_RT.retrieve(
            knowledgeBaseId=KB_ID,
            retrievalQuery={"text": QUERY},
            retrievalConfiguration={"vectorSearchConfiguration": {"numberOfResults": 5, **cfg}},
        )
        hits = resp["retrievalResults"]
        print(f"{label:10}: {len(hits)} chunks", end="")
        if hits:
            md = hits[0].get("metadata", {})
            print(f"  | top chunk meta: doc_type={md.get('doc_type')}, product_line={md.get('product_line')}")
        else:
            print()
else:
    print("Skipped — no playbook KB available.")

In [ ]:
# --- Lever: reranking ---
# A second-stage cross-encoder reorders the retrieved candidates by true relevance
# and puts the best chunk first — countering "lost in the middle". It usually lets
# you pass FEWER chunks at higher accuracy, so reach for it before raising K.
RERANK_MODEL = f"arn:aws:bedrock:{REGION}::foundation-model/amazon.rerank-v1:0"

if KB_ID:
    try:
        resp = AGENT_RT.retrieve(
            knowledgeBaseId=KB_ID,
            retrievalQuery={"text": QUERY},
            retrievalConfiguration={"vectorSearchConfiguration": {
                "numberOfResults": 10,   # retrieve a wider candidate set...
                "rerankingConfiguration": {
                    "type": "BEDROCK_RERANKING_MODEL",
                    "bedrockRerankingConfiguration": {
                        "numberOfRerankedResults": 3,   # ...then keep the best 3
                        "modelConfiguration": {"modelArn": RERANK_MODEL},
                    },
                },
            }},
        )
        print(f"reranked to {len(resp['retrievalResults'])} chunks (best-first):")
        for i, ch in enumerate(resp["retrievalResults"], 1):
            print(f"  {i}. score={ch.get('score', 0):.3f}  {ch['content']['text'][:70]}...")
    except Exception as e:
        print("Reranking not available in this account/region:", str(e)[:120])
        print("(Amazon Rerank must be enabled in Bedrock model access — the API shape above is correct.)")
else:
    print("Skipped — no playbook KB available.")

**What you'll see**: 
- **top-K** 10→3 shrinks retrieved context by roughly half with no quality loss on a well-scoped query.
- **Metadata filtering** narrows to just the policy/laptop docs *before* the vector search — fewer, more precise candidates.
- **Reranking** retrieves a wide set then keeps the best 3 by cross-encoder score (best-first), which is why you reach for it before raising K.

The production order of operations: **filter → retrieve → rerank → trim K** — each step makes the context smaller and more relevant. Confirm the trade-offs on YOUR eval set.

---

## 4. Prompt Compression — Lever 10

Compression treats your prompt as a token stream with uneven information density. A small "compression model" — **LLMLingua-2**, a BERT-class token classifier — scores each token's value and drops the low-signal ones. The main model sees a shorter prompt, returns the same answer, you pay less.

| Approach | Compresses to | Accuracy |
|---|---|---|
| Original prompt | 100% | baseline |
| Selective Context (2023) | ~60% | ~baseline |
| LLMLingua (2023) | ~33% | slight drop |
| **LLMLingua-2 (2024)** | **~22%** | **↑ on some tasks** |

LLMLingua-2 is the frontier — task-agnostic, ~1/4 the tokens, and on some benchmarks the *compressed* prompt beats the original (removed tokens were noise). The win scales with how much dynamic context you push: at high RAG volume, dropping chunks to ~1/3 is a direct ~3× cut on that slice of the input bill.

<div class="alert alert-block alert-warning">
<b>The cache gotcha — the load-bearing rule.</b> Compression rewrites the prompt, and cache keys match the longest token prefix from position 0. A rewritten prefix is a permanent cache miss. So the decision order is: <b>cache the stable prefix → use context editing for tool-heavy loops → compress only the remaining large, dynamic, un-cacheable tail</b> (RAG chunks, user docs). Never compress a cached system prompt.
</div>

<div class="alert alert-block alert-info">
<b>LLMLingua-2 runs in-process — no endpoint required.</b> <code>pip install llmlingua</code> pulls in <code>transformers</code> + <code>torch</code>; <code>PromptCompressor(...)</code> downloads the BERT-class model (~500 MB on first use) straight into the kernel and compresses locally — no SageMaker, no API call. It's a <b>demo</b> here only because of that one-time model download; the cell runs the real library if it's installed and otherwise prints the principle and skips cleanly. (In <i>production</i> you'd often host it on a GPU SageMaker/ECS endpoint so the compression latency doesn't block your app — an optional scaling choice, not a prerequisite.)
</div>

In [ ]:
# LLMLingua-2 on a RAG chunk. Compress ONLY this dynamic content — never a cached prefix.
RAG_CHUNK = (
    "The 30-day return policy applies to all undamaged products that include their original packaging. "
    "Customers may initiate a return online by visiting the support portal and entering their order number. "
    "Once the return is approved, a prepaid shipping label will be emailed within one business day. "
    "Refunds are processed within 5-7 business days after the returned item is received and inspected. "
    "Items damaged in transit are replaced free of charge once photos of the damage are submitted."
)

try:
    from llmlingua import PromptCompressor

    # Loaded once at startup and reused across requests (BERT-class, small + fast).
    compressor = PromptCompressor(
        model_name="microsoft/llmlingua-2-xlm-roberta-large-meetingbank",
        use_llmlingua2=True,
        device_map="cpu",
    )
    result = compressor.compress_prompt(
        RAG_CHUNK,
        rate=0.33,                            # keep ~1/3 of tokens
        force_tokens=["\n", ".", "?", "!"],   # never drop these structural tokens
    )
    print(f"origin tokens:     {result['origin_tokens']}")
    print(f"compressed tokens: {result['compressed_tokens']}  ({result['rate']})")
    print(f"\noriginal prompt:\n{RAG_CHUNK}")
    print(f"\ncompressed prompt:\n{result['compressed_prompt']}")
    print("\n-> feed result['compressed_prompt'] as the volatile tail, AFTER your cachePoint.")
except ImportError:
    print("llmlingua not installed — `pip install llmlingua` to run this live.")
    print("Principle: LLMLingua-2 keeps high-signal tokens and drops filler,")
    print("compressing RAG context to ~1/3 its size for the same answer — applied only")
    print("to the dynamic tail, never to a cached system prompt.")

**What you'll see**: the compressed prompt keeps the high-signal tokens and drops filler — same answer, roughly a third of the tokens. The rule that matters either way: compress only dynamic content (RAG chunks, user docs), never a cached prefix. For real workloads: [LLMLingua repo](https://github.com/microsoft/LLMLingua).

---

## 5. Conversation & Memory — Lever 11

A multi-turn agent leaks context in two directions, and they're the same discipline at two timescales: **decide what enters the context window, and when.**

- **Within a session** — the naive approach replays the whole history every turn. By turn 50 each request carries 49 prior turns, so per-turn cost climbs linearly until you hit the context window and the model errors out.
- **Across sessions** — the agent starts cold tomorrow and re-asks for preferences it already learned.

The fix within a session is **sliding window + rolling summary**: keep the last N messages verbatim, fold everything older into a running summary, so per-turn cost stays roughly constant *and* the important facts survive. The knobs:

| Knob | Typical | Tune by |
|---|---|---|
| Window size N | 5-10 turns | Model loses recent context → raise N; cost creeps → lower N |
| Summary refresh | Every turn past the window, or a token threshold | Per-turn is simplest; threshold-based thrashes the cache less |
| Summary model | A cheaper model than the main one | Small/fast for the summary, the workhorse for the reply |

A conversation manager in Strands is just a class with an `apply_management(agent)` hook the SDK calls after **every** turn. Strands ships a `SlidingWindowConversationManager` (bound by message count, but it *deletes* old turns) and a `SummarizingConversationManager` (summarizes, but only when the context actually overflows). To get exactly "keep the last N, summarize the rest, every turn," we'll first watch the bare window **bound and forget**, then write a ~20-line manager that bounds **and** remembers. It's **purely client-side** — no AgentCore Memory, no AWS resource or role, same Bedrock calls.

<div class="alert alert-block alert-warning">
<b>The cache interaction.</b> Trimming history modifies the prompt prefix, invalidating downstream cache breakpoints (Lever 04). Cache up to the stable <i>summary</i>, leave only the recent N turns dynamic, and refresh the summary on a threshold rather than every single turn so you're not busting the cache constantly.
</div>

In [ ]:
# First, SEE the problem: naive full-history vs a sliding window, as a conversation grows.
# A realistic multi-issue support conversation (12 rounds).
CONVO_TURNS = [
    ("Hi, the monitor from order #A-12345 has a dead pixel.", "Sorry about that — when did it arrive?"),
    ("About three weeks ago.", "You're within the 30-day window: full refund or replacement?"),
    ("Replacement, same model.", "Done. Is the shipping address on file still correct?"),
    ("Yes, same address.", "Great. I'll queue the replacement monitor."),
    ("Also, my keyboard from order #B-67890 has a sticking spacebar.", "Noted — is it within warranty?"),
    ("Bought it two months ago.", "That's covered by the 1-year manufacturer warranty."),
    ("Can you ship both together?", "I can bundle the replacement monitor and a warranty keyboard."),
    ("What's the ETA?", "Typically 2-3 business days once processed."),
    ("Do I need to return the old monitor?", "Yes — a prepaid label will be emailed to you."),
    ("And the keyboard?", "Keep using it until the replacement arrives, then return it with the same label."),
    ("Will I get one tracking number or two?", "One combined shipment, one tracking number."),
    ("Perfect, thanks.", "Anything else I can help with today?"),
]

def to_messages(turns):
    msgs = []
    for u, a in turns:
        msgs.append({"role": "user", "content": [{"text": u}]})
        msgs.append({"role": "assistant", "content": [{"text": a}]})
    return msgs

def est_tokens(messages):
    return sum(len(c["text"]) for m in messages for c in m["content"]) // 4  # rough

print(f"{'turn':>4}  {'full_history':>13}  {'sliding(4)':>11}")
for t in [1, 4, 8, 12]:
    so_far = CONVO_TURNS[:t]
    full = est_tokens(to_messages(so_far))
    slide = est_tokens(to_messages(so_far[-4:]))
    print(f"{t:>4}  {full:>13}  {slide:>11}")
print("\n-> full history grows every turn; a window stays flat — but a bare window forgets.")

**What you'll see**: full history grows every turn while a sliding window stays flat. But a bare window *forgets* — once early turns scroll out, the agent loses the order number mentioned there. The next cell shows that forgetting live, and the one after fixes it by summarizing the dropped turns instead of deleting them.

In [ ]:
# Sliding window, live. window_size caps the message list; per_turn=True trims proactively
# after every turn (not just on overflow). callback_handler=None silences the streamed
# replies so we only print what we're measuring. (No tools here — we're demonstrating
# context management, so we keep the message stream clean and predictable.)
from strands import Agent
from strands.models import BedrockModel
from strands.agent.conversation_manager import SlidingWindowConversationManager

window = SlidingWindowConversationManager(window_size=6, per_turn=True)
windowed_agent = Agent(
    model=BedrockModel(model_id=HAIKU, region_name=REGION),
    conversation_manager=window,
    system_prompt="You are a concise customer-support agent.",
    callback_handler=None,
)

convo = [
    "Hi — my monitor from order #A-12345 has a dead pixel.",   # order # lives in turn 1
    "I'd like a replacement, same model.",
    "Separately, my keyboard from order #B-67890 has a sticking spacebar.",
    "Can you ship the replacement monitor and the keyboard fix together?",
    "What's the ETA on that combined shipment?",
]

# window_size=6 keeps the last 3 rounds (each round = 1 user + 1 assistant = 2 messages).
print("turn | messages retained (window_size=6)")
for i, turn in enumerate(convo, 1):
    windowed_agent(turn)
    print(f"  {i:>2} | {len(windowed_agent.messages)}")

# By now turn 1 has scrolled out of the 6-message window, so the agent can't recall #A-12345.
probe = windowed_agent("Can you remind me the order number for my monitor, for my records?")
print("\nQ:", "Can you remind me the order number for my monitor, for my records?")
print("A:", probe.message["content"][0]["text"])

**What you'll see**: the message count climbs to 6 and then stays flat — the window is bounded. But the cost is memory: turn 1 scrolled out, so the agent can't recall order **#A-12345**. A bare window trades recall for a fixed context size.

The fix is to **summarize the turns that fall out of the window instead of dropping them**. The next cell is a small custom `ConversationManager`: after every turn it keeps the last 6 messages verbatim and folds everything older into a running summary that always preserves identifiers (order numbers, names, plan). Same bounded size — but now the facts survive.

In [ ]:
# Sliding window + rolling summary, as one small manager. apply_management() runs after
# every turn: keep the last `keep` messages verbatim, fold everything older into a running
# summary that always preserves identifiers. This is the "bound AND remember" fix.
from strands.agent.conversation_manager.conversation_manager import ConversationManager

class SlidingWindowWithSummary(ConversationManager):
    def __init__(self, keep=6, summarizer=None):
        super().__init__()
        self.keep = keep
        self.summarizer = summarizer
        self.summary = ""

    def apply_management(self, agent, **kwargs):
        msgs = agent.messages
        if len(msgs) <= self.keep:
            return  # still under the window — nothing to fold yet
        old, recent = msgs[:-self.keep], msgs[-self.keep:]
        transcript = "\n".join(
            f"{m['role']}: {m['content'][0].get('text', '')}" for m in old if m.get("content")
        )
        self.summary = self.summarizer(
            f"Running summary so far:\n{self.summary}\n\nNew turns to fold in:\n{transcript}\n\n"
            "Return an updated <=4-bullet summary. Always keep order numbers, names, plan, and preferences."
        ).message["content"][0]["text"]
        # Rewrite history: one summary message, then the recent window verbatim.
        summary_msg = {"role": "user", "content": [{"text": f"[Earlier conversation summary]\n{self.summary}"}]}
        msgs[:] = [summary_msg] + recent

    def reduce_context(self, agent, e=None, **kwargs):
        # Also fold on a hard context overflow (same strategy).
        self.apply_management(agent)

summarizer = Agent(
    model=BedrockModel(model_id=HAIKU, region_name=REGION),
    system_prompt="You compress conversation history. Output only the requested bullets.",
    callback_handler=None,
)
managed_agent = Agent(
    model=BedrockModel(model_id=HAIKU, region_name=REGION),
    conversation_manager=SlidingWindowWithSummary(keep=6, summarizer=summarizer),
    system_prompt="You are a concise customer-support agent.",
    callback_handler=None,
)

# Same 5-turn conversation as the bare window above.
print("turn | messages retained | running-summary chars")
for i, turn in enumerate(convo, 1):
    managed_agent(turn)
    mgr = managed_agent.conversation_manager
    print(f"  {i:>2} | {len(managed_agent.messages):>17} | {len(mgr.summary):>5}")

print("\nRunning summary after the conversation:\n")
print(managed_agent.conversation_manager.summary)

# Same probe the bare window failed — now the monitor's order number lives in the summary.
probe = managed_agent("Can you remind me the order number for my monitor, for my records?")
print("\nQ:", "Can you remind me the order number for my monitor, for my records?")
print("A:", probe.message["content"][0]["text"])

**What you'll see**: the retained message count stays bounded (one summary message + the last 6), the running summary grows to hold the older turns' facts, and — unlike the bare window — the agent now answers **#A-12345** correctly, because that order number lives in the summary even though the verbatim turn scrolled out. Bounded cost *with* memory of what mattered, in ~20 lines.

That's the within-session half: keep recent turns verbatim, summarize the rest, refresh every turn. Remembering Alex *across* sessions is the other half of the lever — **AgentCore Memory**, next.

### Cross-session memory — AgentCore Memory

The sliding window keeps **one session** bounded. The next problem is *across* sessions: when Alex returns tomorrow, you don't want to replay the whole transcript to re-learn that they're on the Pro plan. **AgentCore Memory** is the managed service for this — no infrastructure to run:

- **Short-term** — raw turns of a session, recalled verbatim with `get_last_k_turns`.
- **Long-term** — facts and preferences asynchronously *extracted* from those turns into a searchable vector store, recalled later (in any future session).

You pick which extraction strategies to enable; AgentCore manages the prompt, model, and schema, landing records under a namespace template:

| Strategy | Extracts | Default namespace |
|---|---|---|
| **Semantic** | Standalone facts about the user/world | `/users/{actorId}/facts/` |
| **User preference** | Stable per-user preferences | `/users/{actorId}/preferences/` |
| **Summary** | Rolling conversation summary | `/sessions/{sessionId}/summary/` |
| **Episodic** | Meaningful interaction sequences | `/episodes/{actorId}/` |

We'll do the full lifecycle: **create** a memory resource (you need to know how), write a realistic support conversation as events, recall it short-term, then — in a fresh session — let a **Strands agent** retrieve the extracted facts through the SDK-native `AgentCoreMemoryToolProvider`, the documented bridge between Strands and AgentCore Memory.

<div class="alert alert-block alert-warning">
<b>Creating a memory needs an execution role.</b> AgentCore assumes a role (trust: <code>bedrock-agentcore.amazonaws.com</code> + <code>bedrock:InvokeModel</code>) to run the async extraction. The workshop stack provisions one and stores its ARN in SSM — the cell reads it from there. That role is scoped to memory resources named <code>customersupport*</code>, so the cell names the memory <code>customersupport_playbook</code> — a different name is denied with an access error. Namespaces drive both retrieval and IAM, are <b>hard to migrate later</b>, and extraction is <b>asynchronous and not free</b> (each pass calls a Bedrock model in your account).
</div>

In [ ]:
import os, time
from bedrock_agentcore.memory import MemoryClient

# AgentCore Memory needs an execution role it ASSUMES to run the async fact-extraction
# pass (trust: bedrock-agentcore.amazonaws.com + bedrock:InvokeModel). The workshop stack
# provisions one and stores its ARN in SSM; we read it from there (or MEMORY_EXECUTION_ROLE_ARN).
MEMORY_ROLE_ARN = os.environ.get("MEMORY_EXECUTION_ROLE_ARN", "")
if not MEMORY_ROLE_ARN:
    try:
        ssm = boto3.client("ssm", region_name=REGION)
        MEMORY_ROLE_ARN = ssm.get_parameter(
            Name="/app/customersupport/agentcore/runtime_iam_role"
        )["Parameter"]["Value"]
    except Exception as e:
        print("Could not read the execution-role ARN from SSM:", e)

MEMORY_READY = False
memory_id = None
ACTOR_ID = "alex-99"                        # stable per end-user; templates the namespace
SESSION_ID = f"sess-{int(time.time())}"     # this conversation

# The memory NAME must start with "customersupport" — the execution role is scoped to
# arn:...:memory/customersupport* (see RuntimeAgentCoreRole), so any other name is denied.
MEMORY_NAME = "customersupport_playbook"

if MEMORY_ROLE_ARN:
    mem = MemoryClient(region_name=REGION)
    print(f"Creating (or reusing) memory '{MEMORY_NAME}' — provisioning takes ~1-2 min the first time...")

    # create_or_get_memory is idempotent: it reuses an existing resource of the same name
    # instead of erroring on re-run. The semantic strategy extracts durable user facts
    # into the {actorId}-templated namespace asynchronously after each event.
    memory = mem.create_or_get_memory(
        name=MEMORY_NAME,
        description="Conversation memory for Lever 11 hands-on",
        strategies=[
            {
                "semanticMemoryStrategy": {
                    "name": "UserFacts",
                    "namespaces": ["/users/{actorId}/facts"],
                }
            }
        ],
        event_expiry_days=7,
        memory_execution_role_arn=MEMORY_ROLE_ARN,
    )
    memory_id = memory["id"]
    MEMORY_READY = True
    print(f"Ready: memory_id={memory_id}  status={memory.get('status', 'n/a')}")
else:
    print("No execution-role ARN found — skipping the AgentCore Memory cells.")
    print("Deploy the workshop stack, or export MEMORY_EXECUTION_ROLE_ARN, to run this section.")

In [ ]:
# Write a realistic support conversation as events. AgentCore stores them short-term
# immediately and queues them for async long-term fact extraction.
CONVO = [
    ("Hi, I'm Alex. My monitor from order #A-12345 has a dead pixel.", "USER"),
    ("Sorry about that, Alex — it's within the 30-day window, so you can get a replacement.", "ASSISTANT"),
    ("Yes please, same model. I'm on the Pro plan, by the way.", "USER"),
    ("Done — a Pro-plan replacement for #A-12345 is queued to your address on file.", "ASSISTANT"),
    ("Great. I usually prefer email updates over phone calls.", "USER"),
    ("Noted — I'll send tracking by email once it ships.", "ASSISTANT"),
]
if MEMORY_READY:
    mem.create_event(memory_id=memory_id, actor_id=ACTOR_ID, session_id=SESSION_ID, messages=CONVO)
    print(f"Wrote {len(CONVO)} messages for actor '{ACTOR_ID}' in session '{SESSION_ID}'.")
    print("Durable facts in here: Pro plan, email-updates preference, open replacement for #A-12345.")
else:
    print("Memory not provisioned — skipped.")

In [ ]:
# SHORT-TERM recall: the exact recent turns of THIS session, returned verbatim and
# instantly (no extraction needed). This is the raw transcript, not distilled facts.
if MEMORY_READY:
    turns = mem.get_last_k_turns(memory_id=memory_id, actor_id=ACTOR_ID, session_id=SESSION_ID, k=5)
    print(f"Short-term: last {len(turns)} turns of session '{SESSION_ID}' (verbatim)\n")
    for n, turn in enumerate(turns, 1):
        for msg in turn:
            print(f"  [{n}] {msg['role']:9} {msg['content']['text']}")
else:
    print("Memory not provisioned — skipped.")

<div class="alert alert-block alert-warning">
<b>Long-term extraction is asynchronous.</b> Short-term recall above is instant. Long-term facts, though, are distilled by an async LLM pass after <code>create_event</code> — so a retrieve right after writing may return nothing. The next cell waits ~60s, then has a <b>Strands agent</b> retrieve via the AgentCore Memory tool in a fresh session. If it still comes back empty, extraction just hasn't finished — re-run the cell; in production you read whatever facts already exist and let the store catch up in the background.
</div>

In [ ]:
# LONG-TERM retrieve — in a FRESH session — via the Strands-native AgentCore Memory tool.
# AgentCoreMemoryToolProvider exposes retrieve over the EXISTING memory; we hand a Strands
# agent the tool and a NEW session_id (Alex "returning tomorrow"), so the only way it can
# know anything about Alex is by retrieving the extracted facts.
if MEMORY_READY:
    from strands import Agent
    from strands.models import BedrockModel
    from strands_tools.agent_core_memory import AgentCoreMemoryToolProvider

    # Extraction is asynchronous — wait for the async pass to distill facts into the namespace.
    print("Waiting ~60s for async fact-extraction to populate the namespace...")
    time.sleep(60)

    provider = AgentCoreMemoryToolProvider(
        memory_id=memory_id,
        actor_id=ACTOR_ID,                            # same user...
        session_id=f"returning-{int(time.time())}",   # ...brand-new session
        namespace=f"/users/{ACTOR_ID}/facts",
        region=REGION,
    )
    returning_agent = Agent(
        model=BedrockModel(model_id=HAIKU, region_name=REGION),
        tools=provider.tools,
        system_prompt=(
            "A returning user just connected. BEFORE replying, use the agent_core_memory "
            "tool to retrieve what you already know about them, then greet them with it."
        ),
        callback_handler=None,   # suppress streamed tool-use chatter; we print the final greeting
    )
    result = returning_agent("Hi, it's Alex again.")
    print("\nGreeting (built from retrieved long-term facts, NOT the transcript):\n")
    print(result.message["content"][0]["text"])
else:
    print("Memory not provisioned — skipped.")

In [ ]:
# Cleanup: delete the memory resource so it stops accruing.
if MEMORY_READY:
    mem.delete_memory(memory_id=memory_id)
    print("Deleted:", memory_id)
else:
    print("Memory not provisioned — nothing to clean up.")

**What you'll see**: short-term recall returns the six turns verbatim immediately. Then, in a brand-new session, the Strands agent calls the `agent_core_memory` tool and greets Alex with what it retrieved — the Pro plan, the email preference, the open #A-12345 replacement — *without* replaying the transcript. That's the cross-session win: a handful of extracted facts instead of the whole history. (If the greeting says it has nothing yet, extraction is still running — re-run the cell.)

This is the SDK-native path: `SummarizingConversationManager` bounds cost *within* a session (client-side), and `AgentCoreMemoryToolProvider` carries facts *across* sessions (managed) — you wire both into a Strands `Agent` rather than hand-rolling either.

---

## 6. Batch Inference — Lever 12

If a workload doesn't need a synchronous answer, you can pay half. Bedrock **batch inference** runs requests asynchronously at **50% of the on-demand token price**, SLA ≤24h (small jobs often finish in minutes). It's file-based: build a JSONL of requests, upload to S3, create a *model invocation job*, collect results from S3. The highest-ROI lever for offline work — and the most underused, because teams default to synchronous calls.

| Workload | Batch fit |
|---|---|
| Backfilling embeddings for an existing corpus | ✅ Perfect |
| Nightly summarization of closed tickets | ✅ Perfect |
| Eval runs against a large test set | ✅ Perfect |
| Bulk classification on a CSV export | ✅ Perfect |
| Real-time chat / tool-calling agents | ❌ No — sync only |

<div class="alert alert-block alert-warning">
<b>What batch does NOT support:</b> no tool calling, no structured output, no multi-turn — each record is processed independently — and it's on-demand only (no provisioned). If your workflow needs any of those, batch is the wrong path. Also: <b>match results by <code>recordId</code>, not line order</b> (ordering isn't guaranteed), and a single bad record fails only itself (failures land in a separate error output — inspect both). Bedrock requires a <b>100-record minimum</b> per job; the cell repeats a small ticket set to 120 to clear it.
</div>

In [ ]:
import json as _json
from pathlib import Path

# Bedrock batch requires a MINIMUM of 100 records per job. We repeat a small set to clear it.
BASE_TICKETS = [
    "Customer reports monitor flickers under load. Resolved with firmware update. Closed.",
    "Customer received wrong color phone. Replacement shipped. Closed.",
    "Customer asked about return policy on opened headphones. Policy explained, no return needed. Closed.",
    "Customer's order delayed by carrier. Refund issued for shipping. Closed.",
    "Customer requested warranty info on a refurbished tablet. Provided. Closed.",
]
TICKETS = (BASE_TICKETS * 24)[:120]  # 120 records — comfortably over the 100 minimum

records = [
    {
        "recordId": f"ticket-{i:06d}",
        "modelInput": {
            "anthropic_version": "bedrock-2023-05-31",
            "max_tokens": 100,
            "messages": [{
                "role": "user",
                "content": f"Summarize this resolved support ticket in one sentence:\n\n{ticket}",
            }],
        },
    }
    for i, ticket in enumerate(TICKETS)
]

local_path = Path("/tmp/batch-input.jsonl")
with local_path.open("w") as f:
    for r in records:
        f.write(_json.dumps(r) + "\n")

print(f"Wrote {local_path} — {len(records)} records, {local_path.stat().st_size} bytes")
print("\nFirst record:")
print(_json.dumps(records[0], indent=2))

In [ ]:
import os, time, boto3

# This cell is GATED: it only submits the batch job when BATCH_BUCKET and BATCH_ROLE_ARN
# are set. Unset (the default), it skips straight to the explanation below and finishes
# instantly — that's expected, not an error. Set both env vars to run the job for real.
BATCH_BUCKET = os.environ.get("BATCH_BUCKET", "")          # an S3 bucket you can write to
BATCH_ROLE_ARN = os.environ.get("BATCH_ROLE_ARN", "")      # IAM role Bedrock assumes for S3 access

if BATCH_BUCKET and BATCH_ROLE_ARN:
    s3 = boto3.client("s3", region_name=REGION)
    bedrock = boto3.client("bedrock", region_name=REGION)
    prefix = f"batch-demo/{int(time.time())}"

    # 1) Upload the input JSONL to S3
    s3.upload_file(str(local_path), BATCH_BUCKET, f"{prefix}/input.jsonl")
    print(f"Uploaded -> s3://{BATCH_BUCKET}/{prefix}/input.jsonl")

    # 2) Submit the batch job (50% of on-demand price)
    job = bedrock.create_model_invocation_job(
        jobName=f"summaries-{int(time.time())}",
        roleArn=BATCH_ROLE_ARN,
        modelId="anthropic.claude-haiku-4-5-20251001-v1:0",
        inputDataConfig={"s3InputDataConfig": {"s3Uri": f"s3://{BATCH_BUCKET}/{prefix}/input.jsonl"}},
        outputDataConfig={"s3OutputDataConfig": {"s3Uri": f"s3://{BATCH_BUCKET}/{prefix}/output/"}},
    )
    job_arn = job["jobArn"]
    print("Submitted job:", job_arn)

    # 3) Poll until done (small jobs usually finish in a few minutes; SLA is up to 24h)
    while True:
        status = bedrock.get_model_invocation_job(jobIdentifier=job_arn)["status"]
        print("  status:", status)
        if status in ("Completed", "Failed", "Stopped", "Expired"):
            break
        time.sleep(30)

    # 4) Read a few results back from S3
    if status == "Completed":
        out = s3.list_objects_v2(Bucket=BATCH_BUCKET, Prefix=f"{prefix}/output/")
        key = next(o["Key"] for o in out["Contents"] if o["Key"].endswith(".jsonl.out"))
        body = s3.get_object(Bucket=BATCH_BUCKET, Key=key)["Body"].read().decode()
        for line in body.splitlines()[:3]:
            rec = _json.loads(line)
            text = rec["modelOutput"]["content"][0]["text"]
            print(f"  {rec['recordId']}: {text[:90]}")
else:
    print("Set BATCH_BUCKET and BATCH_ROLE_ARN env vars to run the batch job live.")
    print("Flow: upload input.jsonl -> create_model_invocation_job -> poll status -> read s3 output.")
    print("Batch runs at 50% of on-demand price; SLA up to 24h (small jobs finish in minutes).")

**What you'll see**: the first cell writes a 120-record JSONL (over the 100 minimum) and prints one record so you can see the `recordId` + `modelInput` shape. The second submits the job, polls `Submitted -> InProgress -> Completed`, and reads a few summaries back from S3 — all at 50% of the sync price. Without `BATCH_BUCKET`/`BATCH_ROLE_ARN` set, it prints the flow instead of running live.

---

## What you've done

You've worked through the six MEDIUM-effort levers, each a standalone architectural move:

1. **LLM routing** — classify, then send each request to the cheapest model that handles it
2. **Guardrails** — screen traffic at the input layer (inline + standalone `ApplyGuardrail`) so refused requests never cost generation
3. **RAG / indexing** — retrieve only the top-K chunks the query needs
4. **Prompt compression** — strip low-signal tokens from the dynamic tail (never the cached prefix)
5. **Conversation & memory** — bound per-turn context with a sliding window + rolling summary, persist across sessions with AgentCore Memory
6. **Batch inference** — run offline work asynchronously at half price

These layer on top of the LOW-effort levers — apply them where they fit, measure on your own eval set, and reach for the HIGH-effort tier only once these are exhausted.

**Next**: [`03-high-effort.ipynb`](./03-high-effort.ipynb) for the compound-AI patterns.